In [1]:
# ============================================================
# 0. Setup and raw data loading
# ============================================================

from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd

RANDOM_STATE = 9890
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Change this only if your CSV files are stored in another folder.
DATA_DIR = Path(".")

def find_file(candidate_names, data_dir=DATA_DIR):
    """
    Looks for a file using a short list of possible names.
    This makes the notebook robust to files like school_covariates.csv
    versus school_covariates(2).csv.
    """
    for name in candidate_names:
        path = data_dir / name
        if path.exists():
            return path
    
    raise FileNotFoundError(
        "Could not find any of these files in "
        f"{data_dir.resolve()}:\n" + "\n".join(candidate_names)
    )

PATHS = {
    "school": find_file(["school_covariates.csv", "school_covariates(2).csv"]),
    "district": find_file(["district_covariates.csv", "district_covariates(2).csv"]),
    "train": find_file(["scores_training.csv", "scores_training(2).csv"]),
    "test": find_file(["scores_test.csv", "scores_test(2).csv"]),
}

school_covariates = pd.read_csv(PATHS["school"])
district_covariates = pd.read_csv(PATHS["district"])
scores_training = pd.read_csv(PATHS["train"])
scores_test = pd.read_csv(PATHS["test"])

# Preserve ID and categorical columns as strings.
STRING_COLS = [
    "ASSESSMENT_ID", "SCHOOL", "DISTRICT", "COUNTY",
    "SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "REGION"
]

for df in [school_covariates, district_covariates, scores_training, scores_test]:
    for col in STRING_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string")

print("Loaded files:")
for key, path in PATHS.items():
    print(f"  {key:8s}: {path}")

def raw_summary(name, df):
    return {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "object_or_string_cols": int(
            df.select_dtypes(include=["object", "string"]).shape[1]
        ),
        "numeric_cols": int(df.select_dtypes(include=[np.number]).shape[1]),
    }

summary = pd.DataFrame([
    raw_summary("school_covariates", school_covariates),
    raw_summary("district_covariates", district_covariates),
    raw_summary("scores_training", scores_training),
    raw_summary("scores_test", scores_test),
])

print("\nRaw table summary:")
print(summary.to_string(index=False))

print("\nKey checks:")
print("school_covariates['SCHOOL'] unique:      ", school_covariates["SCHOOL"].is_unique)
print("district_covariates['DISTRICT'] unique:  ", district_covariates["DISTRICT"].is_unique)
print("scores_training['ASSESSMENT_ID'] unique: ", scores_training["ASSESSMENT_ID"].is_unique)
print("scores_test['ASSESSMENT_ID'] unique:     ", scores_test["ASSESSMENT_ID"].is_unique)

print("\nTarget checks:")
print("'PERCENT_PROFICIENT' in training:", "PERCENT_PROFICIENT" in scores_training.columns)
print("'PERCENT_PROFICIENT' in test:    ", "PERCENT_PROFICIENT" in scores_test.columns)

train_school_coverage = scores_training["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()
test_school_coverage = scores_test["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()

school_districts = school_covariates[["SCHOOL", "DISTRICT"]].drop_duplicates()

train_district_coverage = (
    scores_training[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

test_district_coverage = (
    scores_test[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

print("\nJoin coverage:")
print(f"training rows with SCHOOL in school_covariates: {train_school_coverage:.4f}")
print(f"test rows with SCHOOL in school_covariates:     {test_school_coverage:.4f}")
print(f"training unique schools with DISTRICT data:     {train_district_coverage:.4f}")
print(f"test unique schools with DISTRICT data:         {test_district_coverage:.4f}")

print("\nTarget summary:")
print(scores_training["PERCENT_PROFICIENT"].describe().to_string())

print("\nSoftware:")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Random state:", RANDOM_STATE)

Loaded files:
  school  : school_covariates.csv
  district: district_covariates.csv
  train   : scores_training.csv
  test    : scores_test.csv

Raw table summary:
              table   rows  cols  duplicate_rows  missing_cells  object_or_string_cols  numeric_cols
  school_covariates   4754    52               0          26460                      5            47
district_covariates    674     6               0              0                      1             5
    scores_training 144921     6               0              0                      4             2
        scores_test  48307     5               0              0                      4             1

Key checks:
school_covariates['SCHOOL'] unique:       True
district_covariates['DISTRICT'] unique:   True
scores_training['ASSESSMENT_ID'] unique:  True
scores_test['ASSESSMENT_ID'] unique:      True

Target checks:
'PERCENT_PROFICIENT' in training: True
'PERCENT_PROFICIENT' in test:     False

Join coverage:
training rows with 

In [2]:
# ============================================================
# 1. Merge datasets
# ============================================================

# Merge school covariates
train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

# Merge district covariates
train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

print("train_full shape:", train_full.shape)
print("test_full shape:", test_full.shape)

train_full shape: (144921, 62)
test_full shape: (48307, 61)


In [3]:
# ============================================================
# 2. Missingness overview
# ============================================================

def missing_report(df):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    
    report = pd.DataFrame({
        "missing_count": miss,
        "missing_pct": (miss / len(df)) * 100
    })
    
    return report

train_missing = missing_report(train_full)
test_missing = missing_report(test_full)

print("Train missing columns:", train_missing.shape[0])
print("Test missing columns:", test_missing.shape[0])

print("\nTop 15 missing (train):")
print(train_missing.head(15))

print("\nTop 15 missing (test):")
print(test_missing.head(15))

Train missing columns: 52
Test missing columns: 52

Top 15 missing (train):
                                                    missing_count  missing_pct
TEACHER_TURNOVER_RATE                                      134136    92.558014
KINDERGARTEN_AVERAGE_CLASS_SIZE                            103467    71.395450
GRADE_1_AVERAGE_CLASS_SIZE                                 102899    71.003512
GRADE_2_AVERAGE_CLASS_SIZE                                 102779    70.920709
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_...          89314    61.629439
PERCENT_DROPOUT                                             16061    11.082590
PERCENT_GED                                                 16061    11.082590
PERCENT_STILL_ENROLLED                                      16061    11.082590
PERCENT_NON_DIPLOMA                                         16061    11.082590
PERCENT_DIPLOMA                                             16061    11.082590
SCIENCE_AVERAGE_CLASS_SIZE                             

In [4]:
# ============================================================
# 3. School-level missingness check
# ============================================================

# Example column: ATTENDANCE_RATE (you can change later)
col = "ATTENDANCE_RATE"

train_missing_schools = train_full[train_full[col].isna()]["SCHOOL"].nunique()
test_missing_schools = test_full[test_full[col].isna()]["SCHOOL"].nunique()

train_missing_rows = train_full[col].isna().sum()
test_missing_rows = test_full[col].isna().sum()

print("Using column:", col)

print("\nTrain rows missing:", train_missing_rows)
print("Train unique schools missing:", train_missing_schools)

print("\nTest rows missing:", test_missing_rows)
print("Test unique schools missing:", test_missing_schools)

Using column: ATTENDANCE_RATE

Train rows missing: 3036
Train unique schools missing: 94

Test rows missing: 983
Test unique schools missing: 93


In [6]:
# ============================================================
# 4. Build X and y + preserve IDs
# ============================================================

TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# Save IDs separately (needed for submission later)
train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()

# Target
y_train = train_full[TARGET].copy()

# Drop target from features
X_train = train_full.drop(columns=[TARGET])
X_test = test_full.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (144921, 63)
X_test shape: (48307, 63)
y_train shape: (144921,)


In [7]:
# ============================================================
# 5. Column typing
# ============================================================

ID_COL = "ASSESSMENT_ID"

# Identify column types
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

# Remove ID from categorical if present
if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

# High-cardinality (simple rule: > 50 unique values)
high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))
print("High-cardinality cols:", high_cardinality_cols)
print("Low-cardinality cols:", low_cardinality_cols)

Numeric cols: 55
Categorical cols: 7
High-cardinality cols: ['SCHOOL', 'DISTRICT', 'COUNTY']
Low-cardinality cols: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'REGION']


In [8]:
# ============================================================
# 6A. Frequency encoding (safe, no leakage)
# ============================================================

X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

freq_encoding_cols = []

for col in high_cardinality_cols:
    freq_map = X_train_proc[col].value_counts(dropna=False)
    
    new_col = col + "_freq"
    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)
    
    freq_encoding_cols.append(new_col)

print("Added frequency columns:", freq_encoding_cols)
print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)

Added frequency columns: ['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']
X_train_proc shape: (144921, 66)
X_test_proc shape: (48307, 66)


In [ ]:
# ============================================================
# 6B. Missing indicators + median imputation (more robust than mean)
# ============================================================

numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_indicator_cols = []

for col in numeric_cols_extended:
    if X_train_proc[col].isna().sum() > 0:
        new_col = col + "_missing"
        
        X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
        X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
        
        median_val = X_train_proc[col].median()
        
        X_train_proc[col] = X_train_proc[col].fillna(median_val)
        X_test_proc[col] = X_test_proc[col].fillna(median_val)
        
        missing_indicator_cols.append(new_col)

print("Missing indicators added:", len(missing_indicator_cols))
print("New shape:", X_train_proc.shape)

Missing indicators added: 52
New shape: (144921, 118)


/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42863/1160919616.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42863/1160919616.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42863/1160919616.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

In [10]:
# ============================================================
# 6C. Drop raw high-cardinality categorical columns
# ============================================================

X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

print("Shape after dropping high-cardinality cols:", X_train_proc.shape)

Shape after dropping high-cardinality cols: (144921, 115)


### Feature Encoding: High-Cardinality Variables

The raw high-cardinality categorical variables `SCHOOL`, `DISTRICT`, and `COUNTY` were removed from the modeling matrix after frequency encodings were created for them.

This prevents the model from directly using raw ID-like categorical labels while still preserving useful information about how frequently each school, district, or county appears in the training data.

After dropping these raw columns, the feature matrix has 115 columns.

In [11]:
# ============================================================
# 6D. One-hot encode low-cardinality categorical variables
# ============================================================

X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

# Align columns (important!)
X_train_proc, X_test_proc = X_train_proc.align(X_test_proc, join="left", axis=1, fill_value=0)

print("Final X_train shape:", X_train_proc.shape)
print("Final X_test shape:", X_test_proc.shape)

Final X_train shape: (144921, 165)
Final X_test shape: (48307, 165)


### Final Feature Matrix

After full preprocessing:

- High-cardinality variables (`SCHOOL`, `DISTRICT`, `COUNTY`) were replaced with frequency encodings
- Missing values were handled via:
  - median imputation
  - explicit missingness indicator variables
- Low-cardinality categorical variables were one-hot encoded:
  - `SUBGROUP_NAME`, `ASSESSMENT_NAME`, `DISTRICT_TYPE`, `REGION`

Final dimensions:
- Training set: 144,921 rows × 165 features
- Test set: 48,307 rows × 165 features

The feature space is now fully numeric, aligned between train and test, and ready for modeling.

In [13]:
# ============================================================
# Create modeling copy WITHOUT ID (non-destructive)
# ============================================================

X_train_proc_model = X_train_proc.drop(columns=["ASSESSMENT_ID"])
X_test_proc_model = X_test_proc.drop(columns=["ASSESSMENT_ID"])

print("Modeling shape:", X_train_proc_model.shape)

Modeling shape: (144921, 164)


### Modeling Dataset

A separate modeling dataset was created by removing the identifier column `ASSESSMENT_ID`.

Final modeling dimensions:
- Training set: 144,921 rows × 164 features

This ensures that all features used for modeling are numeric or boolean, and that no identifier-based leakage occurs.

In [15]:
# ============================================================
# 7A. Simple Linear Regression (one feature at a time)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

simple_lr_results = []

for col in X_train_proc_model.columns:
    model = LinearRegression()
    model.fit(X_tr[[col]], y_tr)
    
    train_pred = model.predict(X_tr[[col]])
    val_pred = model.predict(X_val[[col]])
    
    simple_lr_results.append({
        "feature": col,
        "train_mse": mean_squared_error(y_tr, train_pred),
        "val_mse": mean_squared_error(y_val, val_pred)
    })

simple_lr_results = pd.DataFrame(simple_lr_results).sort_values("val_mse")

print(simple_lr_results.head(30).to_string(index=False))

                                                    feature  train_mse    val_mse
                         PERCENT_ECONOMICALLY_DISADVANTAGED 570.735244 581.066288
                                         PERCENT_FREE_LUNCH 575.082361 584.868424
                                            PERCENT_DIPLOMA 621.742550 626.029451
                                     PERCENT_STILL_ENROLLED 637.471784 641.000299
                                           PERCENT_HOMELESS 640.796399 644.070716
                                              PERCENT_BLACK 648.963234 652.648495
                                  PERCENT_WITH_DISABILITIES 650.316873 653.054139
                           PERCENT_ENGLISH_LANGUAGE_LEANERS 648.460937 655.348885
                                            ATTENDANCE_RATE 650.513825 657.793054
                                              PERCENT_WHITE 653.127789 658.601548
                                            PERCENT_DROPOUT 653.769449 661.927501
                

In [17]:
# ============================================================
# Extract best simple linear regression feature properly
# ============================================================

best_feature_row = simple_lr_results.loc[simple_lr_results["val_mse"].idxmin()]

print("Best single-feature model:")
print(best_feature_row)

Best single-feature model:
feature      PERCENT_ECONOMICALLY_DISADVANTAGED
train_mse                            570.735244
val_mse                              581.066288
Name: 43, dtype: object


### Simple Linear Regression Baseline

Each feature was tested individually in a simple linear regression model. This creates a baseline ranking of single predictors before fitting larger multiple regression models.

The goal is not to select the final model from one feature, but to identify which variables have the strongest individual linear relationship with `PERCENT_PROFICIENT`.

### Best Simple Linear Regression Feature

The best single-feature model was:

- Feature: `PERCENT_ECONOMICALLY_DISADVANTAGED`
- Train MSE: 570.74  
- Validation MSE: 581.07  

Interpretation:
- Socioeconomic disadvantage is the strongest standalone predictor of `PERCENT_PROFICIENT`.
- However, the error (~581) is substantially higher than the multiple linear regression model (~312), indicating that no single variable explains the outcome well.
- This confirms that predictive power in the dataset is distributed across multiple correlated features rather than dominated by a single factor.

Conclusion:
Simple linear regression provides insight into marginal relationships but is insufficient for accurate prediction on its own.

### Simple Linear Regression Insights

The strongest individual predictors of `PERCENT_PROFICIENT` are:

- Socioeconomic indicators:
  - `PERCENT_ECONOMICALLY_DISADVANTAGED`
  - `PERCENT_FREE_LUNCH`
- Academic outcomes:
  - `PERCENT_DIPLOMA`
  - `PERCENT_STILL_ENROLLED`
- Demographics:
  - `PERCENT_BLACK`, `PERCENT_WHITE`, `PERCENT_HISPANIC`
- Vulnerability indicators:
  - `PERCENT_HOMELESS`, `PERCENT_WITH_DISABILITIES`
- Attendance:
  - `ATTENDANCE_RATE`

Key observations:
- Socioeconomic disadvantage is the strongest single predictor.
- Many top features are highly correlated (e.g., free lunch vs economic disadvantage).
- Missingness indicators appear, confirming that missing data carries signal.
- Frequency-encoded variables are not dominant individually, suggesting entity effects are weaker in isolation.

Conclusion:
Simple linear regression highlights strong marginal relationships but does not account for interactions or multicollinearity.

In [18]:
# ============================================================
# 7B. Multiple Linear Regression (clean baseline)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Fresh split (consistent with 7A)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

model = LinearRegression()
model.fit(X_tr, y_tr)

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_mse = mean_squared_error(y_tr, train_pred)
val_mse = mean_squared_error(y_val, val_pred)

print("Train MSE:", train_mse)
print("Validation MSE:", val_mse)

Train MSE: 305.12409327017724
Validation MSE: 312.60167029207344


### Multiple Linear Regression Baseline

A multiple linear regression model was fitted using all available features.

Results:
- Train MSE: 305.12  
- Validation MSE: 312.60  

Interpretation:
- The gap between training and validation error is small, indicating minimal overfitting.
- The model performs substantially better than the best simple linear regression (~581 MSE), showing that predictive power is distributed across multiple features.
- The relatively low validation error suggests that linear relationships capture a large portion of the underlying structure in the data.

Conclusion:
Multiple linear regression provides a strong and stable baseline for evaluating more complex models.

In [19]:
# Select top 10 features from simple LR
top_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()

print(top_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']


In [20]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_tr_poly = poly.fit_transform(X_tr[top_features])
X_val_poly = poly.transform(X_val[top_features])

print("Poly feature shape:", X_tr_poly.shape)

Poly feature shape: (115936, 65)


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_poly, y_tr)

train_pred = model.predict(X_tr_poly)
val_pred = model.predict(X_val_poly)

print("Polynomial Regression:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Polynomial Regression:
Train MSE: 507.61481933233904
Validation MSE: 515.1911286593601


### Polynomial Regression

Polynomial regression (degree 2) was applied to the top 10 features identified from simple linear regression.

Results:
- Train MSE: 507.61  
- Validation MSE: 515.19  

Interpretation:
- Performance is significantly worse than multiple linear regression (~312 MSE).
- This indicates that restricting the model to a small subset of features removes important predictive information.
- The polynomial expansion does not compensate for the loss of breadth in the feature space.

Conclusion:
The dataset appears to benefit more from combining many features linearly rather than modeling nonlinear relationships among a small subset of variables. Polynomial regression is not effective in this setting.

In [22]:
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()
print(top5_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [23]:
from itertools import combinations

interaction_cols = []

X_tr_int = X_tr.copy()
X_val_int = X_val.copy()

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    
    X_tr_int[new_col] = X_tr[f1] * X_tr[f2]
    X_val_int[new_col] = X_val[f1] * X_val[f2]
    
    interaction_cols.append(new_col)

print("Number of interaction features added:", len(interaction_cols))

Number of interaction features added: 10


In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_int, y_tr)

train_pred = model.predict(X_tr_int)
val_pred = model.predict(X_val_int)

print("Interaction Model:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Interaction Model:
Train MSE: 304.32506767495795
Validation MSE: 311.9555267461479


### Interaction Model

Pairwise interaction terms were added between the top 5 features identified from simple linear regression.

Results:
- Train MSE: 304.33  
- Validation MSE: 311.96  

Interpretation:
- The interaction model slightly improves performance compared to multiple linear regression (~312.60 → ~311.96).
- The improvement is marginal, suggesting that most of the predictive structure is already captured by additive linear effects.
- Interactions contribute some additional signal but are not a dominant factor in this dataset.

Conclusion:
While interaction terms provide a small improvement, the dataset is largely driven by additive relationships rather than strong nonlinear interactions.

In [25]:
# ============================================================
# 8A. Build controlled candidate feature spaces
# ============================================================

from itertools import combinations

# Use strongest marginal predictors as candidates for engineered terms
top10_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()

# Base space
X_tr_base = X_tr.copy()
X_val_base = X_val.copy()

# Base + polynomial squares for top10
X_tr_polyspace = X_tr.copy()
X_val_polyspace = X_val.copy()

poly_cols = []

for col in top10_features:
    new_col = col + "_squared"
    X_tr_polyspace[new_col] = X_tr[col] ** 2
    X_val_polyspace[new_col] = X_val[col] ** 2
    poly_cols.append(new_col)

# Base + pairwise interactions for top5
X_tr_intspace = X_tr.copy()
X_val_intspace = X_val.copy()

interaction_cols = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    X_tr_intspace[new_col] = X_tr[f1] * X_tr[f2]
    X_val_intspace[new_col] = X_val[f1] * X_val[f2]
    interaction_cols.append(new_col)

# Base + polynomial + interactions
X_tr_combined = X_tr_polyspace.copy()
X_val_combined = X_val_polyspace.copy()

for col in interaction_cols:
    X_tr_combined[col] = X_tr_intspace[col]
    X_val_combined[col] = X_val_intspace[col]

feature_spaces = {
    "base": (X_tr_base, X_val_base),
    "base_plus_poly": (X_tr_polyspace, X_val_polyspace),
    "base_plus_interactions": (X_tr_intspace, X_val_intspace),
    "base_plus_poly_interactions": (X_tr_combined, X_val_combined)
}

print("Top 10 features used for squares:")
print(top10_features)

print("\nTop 5 features used for interactions:")
print(top5_features)

print("\nPolynomial columns added:", len(poly_cols))
print("Interaction columns added:", len(interaction_cols))

print("\nFeature space shapes:")
for name, (X_train_space, X_val_space) in feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Top 10 features used for squares:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features used for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

Polynomial columns added: 10
Interaction columns added: 10

Feature space shapes:
base (115936, 164) (28985, 164)
base_plus_poly (115936, 174) (28985, 174)
base_plus_interactions (115936, 174) (28985, 174)
base_plus_poly_interactions (115936, 184) (28985, 184)


In [ ]:
# ============================================================
# 8B. Forward stepwise selection across feature spaces
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def forward_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    remaining = list(X_train_space.columns)
    selected = []
    rows = []
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            val_mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, val_mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
            
            rows.append({
                "num_features": len(selected),
                "feature_added": best_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(rows), selected

forward_summary = {}

for space_name, (X_train_space, X_val_space) in feature_spaces.items():
    results, selected = forward_stepwise(
        X_train_space, X_val_space, y_tr, y_val, max_features=30
    )
    
    forward_summary[space_name] = {
        "results": results,
        "selected_features": selected,
        "best_val_mse": results["val_mse"].min() if len(results) > 0 else None,
        "best_num_features": results.loc[results["val_mse"].idxmin(), "num_features"] if len(results) > 0 else None
    }
    
    print("\n" + "=" * 80)
    print(space_name)
    print("=" * 80)
    print(results.tail(10).to_string(index=False))
    print("Best validation MSE:", forward_summary[space_name]["best_val_mse"])


base
 num_features                                       feature_added    val_mse
           21                                   DISTRICT_TYPE_NYC 356.639984
           22                     ASSESSMENT_NAME_RegentsScience8 352.619203
           23                               ASSESSMENT_NAME_MATH4 348.915113
           24                               PERCENT_REDUCED_LUNCH 346.330326
           25           ASSESSMENT_NAME_Regents Phy Set/Chemistry 344.123928
           26        ASSESSMENT_NAME_Regents Common Core Geometry 341.579203
           27             ASSESSMENT_NAME_Regents Phy Set/Physics 338.762894
           28 HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE 336.466988
           29                               ASSESSMENT_NAME_MATH7 334.174799
           30                               ASSESSMENT_NAME_MATH3 332.314849
Best validation MSE: 332.3148493948422

base_plus_poly
 num_features                                       feature_added    val_mse
           21  

### Forward Stepwise Design Choice

Forward stepwise selection was used to evaluate whether a smaller subset of predictors can approach the performance of the full multiple linear regression model.

Because the full feature space contains 164–184 predictors depending on the feature set, unrestricted stepwise selection would be computationally expensive and would gradually reconstruct the full model. To keep the procedure tractable and focused on model parsimony, the search was capped at 30 selected features.

This cap is not intended to imply that 30 is theoretically optimal. Instead, it provides a practical stopping limit that allows us to examine whether most predictive gains occur early in the selection path. If validation MSE is still improving near 30 features, the cap can be increased later.

### Forward Stepwise Selection

Forward stepwise selection was applied across multiple feature spaces, including:
- Base feature space
- Base + polynomial terms
- Base + interaction terms
- Base + polynomial + interaction terms

Results:
- Best validation MSE (base): 332.31  
- Best validation MSE (poly): 329.66  

Interpretation:
- All stepwise models perform significantly worse than the full multiple linear regression model (~312.60).
- This indicates that predictive performance relies on combining a large number of features rather than selecting a small subset.
- Polynomial and interaction features provide only marginal improvements within the stepwise framework.

Conclusion:
Subset selection via forward stepwise is not effective for this dataset. The data exhibits a high-dimensional additive structure where many weak predictors contribute jointly. Methods that retain all features while controlling complexity (e.g., Ridge or Lasso) are more appropriate.

In [27]:
# ============================================================
# Backward Stepwise (controlled)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def backward_stepwise(X_train_space, X_val_space, y_train, y_val, max_removals=30):
    
    selected = list(X_train_space.columns)
    results = []
    
    # Initial model (full)
    model = LinearRegression()
    model.fit(X_train_space[selected], y_train)
    val_pred = model.predict(X_val_space[selected])
    best_mse = mean_squared_error(y_val, val_pred)
    
    results.append({
        "num_features": len(selected),
        "removed_feature": None,
        "val_mse": best_mse
    })
    
    for _ in range(max_removals):
        candidates = []
        
        for feature in selected:
            trial_features = [f for f in selected if f != feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        worst_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse <= best_mse:
            selected.remove(worst_feature)
            best_mse = candidate_mse
            
            results.append({
                "num_features": len(selected),
                "removed_feature": worst_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(results)


backward_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = backward_stepwise(X_train_space, X_val_space, y_tr, y_val, max_removals=30)
    
    backward_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features                       removed_feature    val_mse
          153                         PERCENT_ASIAN 312.378267
          152          ASSESSMENT_NAME_RegentsMath8 312.378267
          151          ASSESSMENT_NAME_RegentsMath7 312.367797
          150                  REGION_Southern Tier 312.367797
          149                    REGION_Long Island 312.357865
          148                  REGION_Mohawk Valley 312.353135
          147 PERCENT_OF_STUDENTS_SUSPENDED_missing 312.353135
          146               ATTENDANCE_RATE_missing 312.353135
          145                  SUBGROUP_NAME_Female 312.353135
          144                      N_PUPILS_missing 312.353135
Best val MSE: 312.3531354590935

base_plus_poly
 num_features                               removed_feature    val_mse
          167                                   PERCENT_GED 308.801652
          166                                      GRADE_07 308.801547
          165 ASSESSMENT_NAME_Regents Co

In [29]:
# ============================================================
# Hybrid Stepwise (forward + backward cleanup)
# ============================================================

def hybrid_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    
    remaining = list(X_train_space.columns)
    selected = []
    results = []
    
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        
        # Forward step
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
        else:
            break
        
        # Backward cleanup step
        improved = True
        while improved and len(selected) > 1:
            improved = False
            
            for feature in selected:
                trial_features = [f for f in selected if f != feature]
                
                model = LinearRegression()
                model.fit(X_train_space[trial_features], y_train)
                
                val_pred = model.predict(X_val_space[trial_features])
                mse = mean_squared_error(y_val, val_pred)
                
                if mse < best_mse:
                    selected.remove(feature)
                    best_mse = mse
                    improved = True
                    break
        
        results.append({
            "num_features": len(selected),
            "val_mse": best_mse
        })
    
    return pd.DataFrame(results)


hybrid_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = hybrid_stepwise(X_train_space, X_val_space, y_tr, y_val, max_features=30)
    
    hybrid_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly
 num_features    val_mse
           21 352.226262
           22 348.874109
           23 346.326338
           24 343.776219
           25 341.160388
           26 338.657725
           27 336.137043
           28 333.717583
           29 331.600432
           30 329.660042
Best val MSE: 329.6600419738662

base_plus_interactions
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly_interactions
 num

### Backward and Hybrid Stepwise Selection

Backward and hybrid stepwise selection were evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Backward stepwise performed better than forward stepwise because it began with the full model and removed only features whose exclusion improved or preserved validation performance.

Best backward stepwise validation MSE values:

- Base: 312.35
- Base + polynomial: 308.77
- Base + interactions: 311.70
- Base + polynomial + interactions: 308.10

The best backward stepwise model was the base + polynomial + interaction model, with validation MSE of 308.10. This improves on the full multiple linear regression baseline of approximately 312.60.

The hybrid stepwise results matched the forward stepwise results exactly, suggesting that the backward cleanup phase did not remove any features after forward additions. Therefore, in this implementation, hybrid stepwise effectively behaved like forward stepwise.

Interpretation:
- Forward stepwise performed worse because it was limited to 30 selected features and could not capture the distributed signal across many predictors.
- Backward stepwise performed better because it retained most of the full feature space while pruning redundant or harmful variables.
- Polynomial terms provided meaningful improvement when added to the full feature space and pruned through backward selection.
- Interaction terms alone provided only modest improvement.

Conclusion:
The strongest linear-model-family result so far is backward stepwise on the combined polynomial + interaction feature space. This suggests that the dataset is mostly additive and high-dimensional, but selected nonlinear terms can improve performance when incorporated carefully.